In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers


2026-02-15 17:15:05.592803: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# read dataset with comma as decimal separator so numeric columns are parsed correctly
# na_values handled as before

df = pd.read_csv('./AirQualityUCI.csv', sep=';', na_values=-200, decimal=',')

In [3]:
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 9471 entries, 0 to 9470
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9357 non-null   str    
 1   Time           9357 non-null   str    
 2   CO(GT)         7674 non-null   float64
 3   PT08.S1(CO)    8991 non-null   float64
 4   NMHC(GT)       914 non-null    float64
 5   C6H6(GT)       8991 non-null   float64
 6   PT08.S2(NMHC)  8991 non-null   float64
 7   NOx(GT)        7718 non-null   float64
 8   PT08.S3(NOx)   8991 non-null   float64
 9   NO2(GT)        7715 non-null   float64
 10  PT08.S4(NO2)   8991 non-null   float64
 11  PT08.S5(O3)    8991 non-null   float64
 12  T              8991 non-null   float64
 13  RH             8991 non-null   float64
 14  AH             8991 non-null   float64
 15  Unnamed: 15    0 non-null      float64
 16  Unnamed: 16    0 non-null      float64
dtypes: float64(15), str(2)
memory usage: 1.4 MB


In [5]:
df.drop(columns=['Date', 'Time', "Unnamed: 15", "Unnamed: 16"], inplace=True)

In [6]:
df.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [7]:
df.dropna(inplace=True)

In [8]:
df.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [9]:
x = df.drop(columns=['CO(GT)'])
y = df['CO(GT)']

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [11]:
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=[12]),
    layers.Dropout(.3),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.1),
    layers.Dense(1, activation='linear')
])

/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1771168524.574219  185616 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 747 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [12]:

from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping

checkpoint = ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_loss',
    verbose=0,
    save_best_only=True,
    mode='min'
)

earlystop = EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True,
    mode='min',
    verbose=1
)


In [13]:
model.compile(optimizer='adam', loss='mse', metrics=['mae']),

(None,)

In [14]:
model.fit(x_train, y_train, epochs=100, validation_split=0.2, callbacks=[checkpoint, earlystop])

Epoch 1/100


2026-02-15 17:15:29.784949: I external/local_xla/xla/service/service.cc:163] XLA service 0x78ee94003ad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-15 17:15:29.784961: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2026-02-15 17:15:29.979771: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-15 17:15:30.638525: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801


12/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 155904.4056 - mae: 313.8631 

I0000 00:00:1771168534.276674  185877 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 235ms/step - loss: 90510.5156 - mae: 233.8147 - val_loss: 1078.5093 - val_mae: 26.7106
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 37400.8555 - mae: 153.9728 - val_loss: 5295.6162 - val_mae: 71.0702
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 19583.2246 - mae: 112.7501 - val_loss: 808.7595 - val_mae: 25.3709
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 18143.8066 - mae: 105.0837 - val_loss: 2065.0020 - val_mae: 43.1755
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 13860.5576 - mae: 94.2793 - val_loss: 3350.9221 - val_mae: 57.1350
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9120.6553 - mae: 74.6539 - val_loss: 2102.1934 - val_mae: 45.1818
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6569.7461 - mae: 64.9595 - val_loss: 571.7910 - val_mae: 21.8627
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 6273.7422 - mae: 62.9918 - val_loss: 1222.0177 - val_mae: 33.8479


In [15]:
model.predict(x_test)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step


array([[-8.1384277e-01],
       [ 2.2603025e+00],
       [-1.2276495e+00],
       [ 5.4185214e-03],
       [-1.7374709e+00],
       [-9.5250869e-01],
       [ 3.3747613e+00],
       [-2.5702909e-01],
       [-2.5154111e+00],
       [ 4.9607059e-01],
       [-2.2630351e+00],
       [ 1.2471716e+00],
       [-5.8655572e-01],
       [ 1.8649414e+00],
       [ 2.1439512e+00],
       [ 7.3337302e-02],
       [-1.4021184e+00],
       [ 1.6325328e+00],
       [ 3.2673500e+00],
       [-2.6671166e+00],
       [-1.3460710e+00],
       [-1.3106208e+00],
       [-1.6274831e+00],
       [ 3.7208626e-01],
       [-1.5221212e+00],
       [-6.8516040e-01],
       [ 4.4667659e+00],
       [ 5.3646970e-01],
       [-6.5191274e+00],
       [ 4.8436570e+00],
       [ 3.0531538e+00],
       [-2.9740019e+00],
       [-7.8523850e-01],
       [-1.1399517e+00],
       [-2.1488724e+00],
       [ 2.0206645e+00],
       [ 1.0191824e+00],
       [ 1.4319728e-01],
       [ 3.1543605e+00],
       [ 5.5142331e-01],
